# Expression Class Tests
Test the unified Expression class for SYMBA symbolic regression.

In [ ]:
import sys, os

# If running from Colab, clone the repo first
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/SYMBA'):
        !git clone https://github.com/ML4SCI/SYMBA.git /content/SYMBA
    os.chdir('/content/SYMBA/David_ZiPeng_Zhou')
else:
    # Local: ensure symbolic_jepa package is importable
    os.chdir(os.path.dirname(os.path.abspath('__file__')) if '__file__' in dir() else '.')
    if os.path.basename(os.getcwd()) != 'David_ZiPeng_Zhou':
        # If running from repo root or elsewhere, adjust
        candidate = os.path.join(os.getcwd(), 'David_ZiPeng_Zhou')
        if os.path.isdir(candidate):
            os.chdir(candidate)
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

from symbolic_jepa import *
import numpy as np
import matplotlib.pyplot as plt

## 1. Load Feynman Equations

In [ ]:
csv_path = '../SYMBA_REG/SymbolicGPT_Krish_Malik/data/Feynman_csv_edit.csv'
# Adjust path for Colab
if 'google.colab' in sys.modules:
    csv_path = 'SYMBA_REG/SymbolicGPT_Krish_Malik/data/Feynman_csv_edit.csv'

exprs = load_feynman_csv(csv_path)
print(f'Loaded {len(exprs)} Feynman expressions')

In [ ]:
# Inspect a few
for i, e in enumerate(exprs[:5]):
    print(f'[{i}] {e.sympy_expr}')
    print(f'     vars: {[v.name for v in e.variables]} -> {e.var_map}')
    print(f'     prefix: {e.prefix}')
    print()

## 2. Tokenization

In [ ]:
tok = PrefixTokenizer()
print(f'Vocab size: {len(tok)}')
print(f'Vocab: {tok.vocab}')
print()

e = exprs[4]  # F = G*m1*m2 / r^2
ids = e.tokenize(tok)
print(f'Expression: {e.sympy_expr}')
print(f'Prefix:     {e.prefix}')
print(f'Token IDs:  {ids}')
print(f'Decoded:    {tok.decode(ids)}')

# Check for <unk> across all expressions
n_unk = sum(1 for e in exprs if tok.unk_id in e.tokenize(tok))
print(f'\nExpressions with <unk>: {n_unk}/{len(exprs)}')

## 3. Prefix Round-Trip

In [ ]:
# Verify prefix -> sympy round-trip for all expressions
n_ok = 0
failures = []
for i, e in enumerate(exprs):
    try:
        recovered, consts = prefix_to_sympy(e.prefix)
        n_ok += 1
    except Exception as ex:
        failures.append((i, e.prefix, str(ex)))

print(f'Round-trip OK: {n_ok}/{len(exprs)}')
for i, p, err in failures:
    print(f'  FAIL [{i}]: {p} -> {err}')

## 4. Sampling & Evaluation

In [ ]:
# Test sampling for all
n_sample_ok = 0
sample_failures = []
for i, e in enumerate(exprs):
    try:
        cloud = e.sample(200, method='uniform')
        assert cloud.shape == (200, len(e.variables) + 1)
        assert np.isfinite(cloud).all()
        n_sample_ok += 1
    except Exception as ex:
        sample_failures.append((i, str(ex)))

print(f'Sampling OK: {n_sample_ok}/{len(exprs)}')
for i, err in sample_failures:
    print(f'  FAIL [{i}]: {err}')

In [ ]:
# Plot a univariate example
e0 = exprs[0]  # exp(-theta^2/2) / sqrt(2*pi)
cloud = e0.sample(500, method='uniform')

plt.figure(figsize=(8, 4))
plt.scatter(cloud[:, 0], cloud[:, -1], s=3, alpha=0.6)
plt.xlabel(e0.variables[0].name)
plt.ylabel('f')
plt.title(f'{e0.sympy_expr}')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# LHS sampling demo
cloud_lhs = e0.sample(200, method='lhs')
cloud_unif = e0.sample(200, method='uniform')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(cloud_unif[:, 0], cloud_unif[:, -1], s=5)
axes[0].set_title('Uniform random')
axes[1].scatter(cloud_lhs[:, 0], cloud_lhs[:, -1], s=5)
axes[1].set_title('Latin Hypercube')
for ax in axes:
    ax.set_xlabel(e0.variables[0].name)
    ax.set_ylabel('f')
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Synthetic Expressions

In [ ]:
# Create synthetic expressions via from_infix
synth = Expression.from_infix('sin(x) + cos(2*x)')
print(f'expr:    {synth.sympy_expr}')
print(f'vars:    {synth.variables}')
print(f'prefix:  {synth.prefix}')
print(f'tokens:  {synth.tokenize(tok)}')
print()

cloud = synth.sample(500, method='grid')
plt.figure(figsize=(8, 4))
plt.plot(cloud[:, 0], cloud[:, -1])
plt.xlabel('x')
plt.ylabel('f(x)')
plt.title(f'{synth.sympy_expr}')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Multi-variable synthetic
import sympy as sp

synth2 = Expression.from_infix('x**2 + y**2')
print(f'expr:    {synth2.sympy_expr}')
print(f'vars:    {synth2.variables}')
print(f'prefix:  {synth2.prefix}')
print()

cloud = synth2.sample(500, method='uniform')
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2], s=2, alpha=0.5)
ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('f')
ax.set_title(f'{synth2.sympy_expr}')
plt.show()

## 6. Equivalence Checking

In [ ]:
# Two algebraically equivalent forms
a = Expression.from_infix('(x+1)**2')
b = Expression.from_infix('x**2 + 2*x + 1')

print(f'a: {a.sympy_expr}')
print(f'b: {b.sympy_expr}')
print(f'a.prefix: {a.prefix}')
print(f'b.prefix: {b.prefix}')
print(f'Equivalent (numeric):   {a.is_equivalent(b, method="numeric")}')
print(f'Equivalent (symbolic):  {a.is_equivalent(b, method="symbolic")}')
print()

# Non-equivalent
c = Expression.from_infix('x**2 + 1')
print(f'c: {c.sympy_expr}')
print(f'a == c (numeric): {a.is_equivalent(c, method="numeric")}')

## 7. from_sympy Constructor

In [ ]:
import sympy as sp

x, y = sp.symbols('x y')
expr = sp.sin(x) * sp.exp(-y**2)

e = Expression.from_sympy(expr, [
    VarMeta('x', -3.14, 3.14),
    VarMeta('y', -2.0, 2.0),
])

print(f'expr:   {e.sympy_expr}')
print(f'prefix: {e.prefix}')
print(f'sample shape: {e.sample(100).shape}')